# MacroAnchor G10 10Y — V4.1: Q-consistent sovereign-credit pricing audit

**New, exploratory follow-up. V2.3 headline and V3.1 credit buckets are unchanged. This is NOT a validated alpha result.**

## Objective

Does a MacroAnchor rates signal derived from a **CDS-calibrated, credit-sensitive bond pricing equation under Q** change cross-country 10Y selection? Compare it with the **frozen** original MacroAnchor `tp_z_past` on **the same countries and dates**, with unchanged realized 12-month local-rate return proxies and DV01-neutral top-3/bottom-3 weights. This is not a CDS-hedged strategy or an estimate of P-world expected default losses.

## Inputs / first run

1. Download/prepare your exact frozen `g10_panel_model_objects.csv` from V2.3.
2. Use the raw Investing.com country CDS 5Y exports (the original `*GV5Y* Overview.csv` files), **or** one merged `sovereign_cds_5y.csv` with `date,country,cds5y_bp` (alternatively Parquet). You may include an `expanded_macroanchor_model_objects.csv` with genuinely reconstructed parent-model signals for extra countries; optional and not used to change V2.3.
3. Press **Run all**: the **first code cell immediately opens the upload window**. Select every required file **together**. No other user interaction, file picker, or network request occurs after this step.

## Method and deliberately strict limitations

- Flat **Q hazard** calibrated from the *5Y* quoted CDS spread with fixed recovery `R=0.40`: `lambda_Q = s_CDS/(1-R)` under idealized continuous premium and protection legs. This is a fixed 5Y→10Y extrapolation assumption, not market-identified 10Y credit structure.
- Invert **zero-coupon recovery-of-par** defaultable bond pricing under Q to infer a *rates-only zero-yield proxy*. **This is not** `TP - CDS`; it requires a bond pricing equation and its recovery cash flow. The observed government 10Y yield is **not an actual zero-coupon yield** (coupon/curve unavailable). Therefore the resulting level is only a transparent **model-implied diagnostic proxy**.
- Keep MacroAnchor's original physical-measure expectations (derived from `y10-tp`) fixed. No Q probability is interpreted as a P probability.
- CDS contract denomination, sovereign CDS *quanto*, reference obligations, 5Y-vs-10Y tenor basis, liquidity, funding and bond–CDS basis are **not identified** by these files. All interpretations are conditional on those strong assumptions. The new signal is **not called a purified term premium**.
- As-of join is strictly backward (max age 45 days); no historical CDS value is invented for Canada outside its actual 2012–2023 range. Expanding historical standardization uses past original-TP moments; all original strategy choices remain frozen.
- This is a signal-substitution experiment on original *unhedged credit-risk returns*, **not** a rates-only realized P&L or deployable risk-hedged trade.

**V4.1 data-compatibility fix:** `duration10` may be absent from frozen parent CSV. It is reconstructed with the exact V2.3 annuity proxy, in memory only; original SHA and strategy are unchanged.


In [ ]:
# ===== FIRST EXECUTABLE CELL: immediate upload; NO LATER PROMPTS =====
try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError('Upload-first notebook is designed for Google Colab.') from exc
print('UPLOAD NOW: frozen g10_panel_model_objects.csv AND raw Investing.com 5Y sovereign CDS CSVs (or one merged CDS history).')
print('Select all files together in this one window. Optional: expanded_macroanchor_model_objects.csv.')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No files selected. A frozen MacroAnchor panel and historical sovereign CDS are required.')
UPLOADED_NAMES = list(uploaded.keys())
print('Initial upload complete; no further interaction will be requested.')
print('Files:', *UPLOADED_NAMES, sep='\n - ')


In [ ]:
# ===== INVESTING.COM RAW CDS CSV BRIDGE — insert AFTER FIRST UPLOAD, BEFORE PREFLIGHT =====
# Accept several original Investing.com exports (one CDS 5Y sovereign per country).
# Converts them, without additional prompts, into sovereign_cds_5y.csv used by the independent V4 Q-pricing audit.
# Parent MacroAnchor strategy, signals, portfolio rules, and statistics are unchanged.

from pathlib import Path
import re
import unicodedata
import hashlib
import json
import pandas as pd

_INV_ISO2 = {
    'DE':'DEU','US':'USA','GB':'GBR','JP':'JPN','CH':'CHE',
    'AU':'AUS','CA':'CAN','SE':'SWE','NO':'NOR','NZ':'NZL',
    'FR':'FRA','IT':'ITA','ES':'ESP','PT':'PRT','BE':'BEL',
    'NL':'NLD','AT':'AUT','FI':'FIN','IE':'IRL','DK':'DNK',
    'KR':'KOR','MX':'MEX','BR':'BRA',
}
_INV_NAMES = {
    'germany':'DEU','allemagne':'DEU','deutschland':'DEU',
    'united states':'USA','usa':'USA','etats unis':'USA',
    'united kingdom':'GBR','great britain':'GBR','uk':'GBR',
    'japan':'JPN','japon':'JPN','switzerland':'CHE','suisse':'CHE',
    'australia':'AUS','australie':'AUS','canada':'CAN',
    'sweden':'SWE','suede':'SWE','norway':'NOR','norvege':'NOR',
    'new zealand':'NZL','france':'FRA','italy':'ITA','italia':'ITA',
    'spain':'ESP','espana':'ESP','espagne':'ESP','portugal':'PRT',
    'belgium':'BEL','netherlands':'NLD','austria':'AUT',
    'finland':'FIN','ireland':'IRL','denmark':'DNK','south korea':'KOR',
    'mexico':'MEX','brazil':'BRA',
}

def _inv_country_from_filename(filename):
    """Require a country in the original filename: ticker or explicit country name."""
    name = Path(filename).stem.upper()
    codes = re.findall(r'(?<![A-Z0-9])([A-Z]{2})GV5Y[A-Z0-9=]*(?![A-Z0-9])', name)
    mapped = {_INV_ISO2[c] for c in codes if c in _INV_ISO2}
    if len(mapped) > 1:
        raise ValueError(f'Ambiguous country ticker in filename: {filename}')
    if mapped:
        return next(iter(mapped))
    norm = ''.join(ch for ch in unicodedata.normalize('NFKD',name.lower())
                   if not unicodedata.combining(ch))
    norm = re.sub(r'[^a-z0-9]+', ' ', norm).strip()
    matches = {iso for label, iso in _INV_NAMES.items()
               if re.search(r'(?<![a-z0-9])' + re.escape(label) + r'(?![a-z0-9])', norm)}
    if len(matches) == 1:
        return next(iter(matches))
    # Explicit ISO3 naming also supported: DEU_CDS5Y_history.csv
    if ('cds' in norm or '5y' in norm) and norm.split(' ')[0].upper() in set(_INV_ISO2.values()):
        return norm.split(' ')[0].upper()
    raise ValueError(f'Cannot identify one sovereign from filename {filename!r}. '
                     'Preserve the Investing.com ticker (e.g. DEGV5YUSAC Overview.csv), '
                     'or rename it as DEU_CDS5Y_raw.csv.')

def _inv_raw_signature(path):
    """Check header only; never mistake the frozen model, ratings or OAS for CDS quotes."""
    if path.suffix.lower() != '.csv':
        return False
    try:
        header = pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns
        names = {str(c).strip().lower() for c in header}
        return {'date','price'}.issubset(names) and bool(
            names.intersection({'open','high','low','change %','change%'}))
    except Exception:
        return False

def _inv_price_bp(series):
    raw = series.astype(str).str.strip().str.replace('\u00a0', '', regex=False)
    raw = raw.str.replace(r'(?i)\s*(bp|bps|basis\s*points)\s*$', '', regex=True)
    # Investing.com export: 12.51, 1,234.56, or 1.234,56 depending on locale.
    def parse(v):
        v = re.sub(r'\s+', '', v)
        if v in {'', '-', '--', 'nan', 'None'}: return float('nan')
        if ',' in v and '.' in v:
            v = v.replace(',', '') if v.rfind('.') > v.rfind(',') else v.replace('.', '').replace(',', '.')
        elif ',' in v:
            if re.fullmatch(r'[+-]?\d{1,3}(,\d{3})+', v): v = v.replace(',', '')
            else: v = v.replace(',', '.')
        try: return float(v)
        except ValueError: return float('nan')
    return pd.to_numeric(raw.map(parse), errors='coerce')

def _inv_date(series, filename):
    raw = series.astype(str).str.strip()
    # US-style dates are what the actual DEGV5YUSAC export uses.
    candidates = [(f, pd.to_datetime(raw, format=f, errors='coerce'))
                  for f in ('%m/%d/%Y', '%Y-%m-%d', '%d/%m/%Y')]
    best_score = max(s.notna().sum() for _, s in candidates)
    best = [(fmt, s) for fmt, s in candidates if s.notna().sum() == best_score]
    if best_score < max(1, int(.95 * len(raw))):
        raise ValueError(f'{filename}: failed to parse >=95% of Investing.com dates')
    # If two date conventions fit equally well and give different dates, refuse silent ambiguity.
    if len(best) > 1 and any(not best[0][1].equals(s) for _, s in best[1:]):
        raise ValueError(f'{filename}: ambiguous date format. Expected MM/DD/YYYY, like German example.')
    return best[0][1]

def _inv_parse_file(path):
    country = _inv_country_from_filename(path.name)
    raw = pd.read_csv(path, encoding='utf-8-sig')
    columns = {str(c).strip().lower(): c for c in raw.columns}
    dates = _inv_date(raw[columns['date']], path.name)
    spread = _inv_price_bp(raw[columns['price']])
    invalid = dates.isna() | spread.isna() | (spread < 0) | (spread >= 3000)
    if invalid.mean() > .01:
        raise ValueError(f'{path.name}: {invalid.sum()} invalid rows ({invalid.mean():.1%}), '
                         'check locale/units before continuing.')
    df = pd.DataFrame({'date': dates.loc[~invalid], 'country': country,
                       'cds5y_bp': spread.loc[~invalid]})
    if df['date'].dt.to_period('M').nunique() < 12:
        raise ValueError(f'{path.name}: fewer than 12 historical months; snapshot/short export rejected.')
    df = df.sort_values('date').drop_duplicates(['country', 'date'], keep='last')
    print(f'  {path.name} -> {country}: {len(df):,} daily quotes; '
          f'{df.date.min():%Y-%m-%d} to {df.date.max():%Y-%m-%d}')
    return df

# Work ONLY with explicitly uploaded files. No downloading, searching or later dialogs.
_inv_raw_files = [Path(name) for name in UPLOADED_NAMES if _inv_raw_signature(Path(name))]
if _inv_raw_files:
    _inv_other_cds = [name for name in UPLOADED_NAMES
                      if name not in {p.name for p in _inv_raw_files}
                      and ('cds' in name.lower() or 'sovereign_spread' in name.lower())
                      and Path(name).suffix.lower() in {'.csv','.parquet','.pq'}]
    if _inv_other_cds:
        raise ValueError('Upload EITHER raw Investing.com files OR one pre-merged CDS history, not both: '
                         + ', '.join(_inv_other_cds))
    print(f'Converting {len(_inv_raw_files)} Investing.com raw CDS exports:')
    _inv_all = pd.concat([_inv_parse_file(p) for p in _inv_raw_files], ignore_index=True)
    dup = _inv_all.duplicated(['country','date'], keep=False)
    if dup.any():
        diffs = _inv_all.loc[dup].groupby(['country','date'])['cds5y_bp'].agg(lambda x: x.max()-x.min())
        if (diffs > 1e-6).any():
            raise ValueError('Conflicting CDS quotes for the same country/date across uploaded files. '
                             'Remove duplicates or compare instrument currencies.')
    _inv_all = _inv_all.sort_values(['country','date']).drop_duplicates(['country','date'])
    _inv_out = Path('sovereign_cds_5y.csv')
    _inv_all.to_csv(_inv_out, index=False, date_format='%Y-%m-%d', float_format='%.6f')
    _inv_manifest = {
        'source': 'Investing.com user-exported sovereign CDS 5Y daily CSV',
        'files': [{'filename': p.name,
                   'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
                   'country': _inv_country_from_filename(p.name)} for p in _inv_raw_files],
        'normalized_file': _inv_out.name,
        'normalized_sha256': hashlib.sha256(_inv_out.read_bytes()).hexdigest(),
        'normalized_rows': len(_inv_all),
        'normalized_countries': sorted(_inv_all.country.unique().tolist()),
    }
    Path('investing_raw_cds_sources.json').write_text(json.dumps(_inv_manifest,indent=2))
    # Hide raw files from V3's one-CDS-file preflight, expose only the canonical merged file.
    UPLOADED_NAMES = [n for n in UPLOADED_NAMES if n not in {p.name for p in _inv_raw_files}]
    UPLOADED_NAMES.append(_inv_out.name)
    print('Generated:', _inv_out, '|', len(_inv_all), 'quotes |',
          _inv_all.country.nunique(), 'countries; no further input required.')
else:
    print('No Investing.com raw exports detected; using existing normalized CDS input unchanged.')


In [ ]:
# ===== PRE-FLIGHT: the upload just selected is the sole source of input files =====
from pathlib import Path
import json, re, hashlib, warnings
import numpy as np
import pandas as pd
from scipy.optimize import brentq

EXPECTED_PARENT_SHA = '1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6'
OUT = Path('macroanchor_qcredit_v4_outputs')
OUT.mkdir(exist_ok=True)

# Colab's repeated uploads are saved as 'filename (4).csv'. Match on a canonicalized
# basename but READ THE EXACT path that was uploaded, not a stale old file in runtime.
def canonical_basename(name):
    return re.sub(r'\s*\(\d+\)(?=\.[^.]+$)', '', Path(name).name).lower()

def select_exact(name, required=True):
    found = [Path(n) for n in UPLOADED_NAMES if canonical_basename(n) == name.lower()]
    if len(found) > 1:
        raise RuntimeError(f'Multiple versions of {name} uploaded: {[str(x) for x in found]}')
    if required and not found:
        raise RuntimeError(f'Required file missing in FIRST upload: {name}')
    if found and not found[0].is_file():
        raise FileNotFoundError(f'Uploaded file is not readable: {found[0]}')
    return found[0] if found else None

MODEL_PATH = select_exact('g10_panel_model_objects.csv')
EXT_PATH = select_exact('expanded_macroanchor_model_objects.csv', required=False)
CDS_CANDIDATES = [Path(n) for n in UPLOADED_NAMES
    if canonical_basename(n) in {'sovereign_cds_5y.csv','sovereign_cds_5y.parquet','cds.parquet'}]
if len(CDS_CANDIDATES) != 1:
    raise RuntimeError('Upload exactly one merged CDS history or multiple original Investing.com CDS CSVs.')
CDS_PATH = CDS_CANDIDATES[0]
# duration10 is OPTIONAL: the frozen V2.3 also derived it in memory from y10.
REQUIRED_PANEL = {'date','country','y10','tp','tp_z_past','rx'}
header = set(pd.read_csv(MODEL_PATH, nrows=0).columns)
missing = REQUIRED_PANEL - header
if missing:
    raise RuntimeError(f'Frozen panel is missing required fields: {sorted(missing)}')

def file_hash(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

PANEL_SHA = file_hash(MODEL_PATH)
print('PANEL SHA256:',PANEL_SHA)
if PANEL_SHA != EXPECTED_PARENT_SHA:
    warnings.warn('Parent panel does not match archived V2.3 SHA. Provenance mismatch: do NOT treat it as frozen verification.')
else:
    print('Frozen original V2.3 model panel checksum VERIFIED.')
print('PREFLIGHT PASS: panel =',MODEL_PATH,'; CDS =',CDS_PATH)
print('No more uploads, manual inputs, or web requests.')


In [ ]:
# ===== METHOD CONSTANTS (fixed before seeing V4 outcomes) =====
TENOR_Y = 10.0                   # zero-coupon bond proxy maturity in years
CDS_TENOR_Y = 5.0                # only observed sovereign CDS tenor
RECOVERY_BASE = 0.40            # pricing convention assumption, not estimated
RECOVERY_SENSITIVITIES = (0.25, 0.60) # model assumption diagnostics; not optimized
MAX_QUOTE_AGE_DAYS = 45
PANEL_CUTOFF = pd.Timestamp('2026-05-31')
MAX_CDS_BP = 3000.0
H = 12
NW_LAG = 12
MIN_COUNTRIES = 6
K = 3                          # V2.3 headline: unchanged
MIN_PARENT_HISTORY = 120        # 120 past *original* MacroAnchor TP observations
COST_BP_GRID = (0.0, 2.0, 5.0, 10.0)

COUNTRY_ISO = {
    'usa':'USA','united states':'USA','united states of america':'USA',
    'deu':'DEU','germany':'DEU','gbr':'GBR','united kingdom':'GBR',
    'jpn':'JPN','japan':'JPN','can':'CAN','canada':'CAN',
    'che':'CHE','switzerland':'CHE','swe':'SWE','sweden':'SWE',
    'nor':'NOR','norway':'NOR','aus':'AUS','australia':'AUS',
    'nzl':'NZL','new zealand':'NZL','fra':'FRA','france':'FRA',
    'ita':'ITA','italy':'ITA','esp':'ESP','spain':'ESP',
    'bel':'BEL','belgium':'BEL','nld':'NLD','netherlands':'NLD',
    'aut':'AUT','austria':'AUT','prt':'PRT','portugal':'PRT',
    'irl':'IRL','ireland':'IRL','dnk':'DNK','denmark':'DNK',
    'kor':'KOR','south korea':'KOR',
}
def standard_country(x):
    a=str(x).strip().lower()
    return COUNTRY_ISO.get(a, a.upper() if len(a)==3 else 'UNKNOWN:'+a)

def read_cds(path):
    if path.suffix.lower() in {'.parquet','.pq'}:
        d=pd.read_parquet(path)
    else:
        d=pd.read_csv(path)
    norm={str(c).lower().strip().replace(' ','_'):c for c in d.columns}
    def match(*names):
        return next((norm[x] for x in names if x in norm),None)
    dc=match('date','scraped_date','observation_date','timestamp')
    cc=match('country','country_name','sovereign','iso3','entity')
    sc=match('cds5y_bp','cds_5y_bp','spread_bp','spread','last','price')
    if not all([dc,cc,sc]):
        raise ValueError(f'Historical CDS must have date,country,cds5y_bp equivalent: {list(d.columns)}')
    if match('tenor','maturity'):
        ten=match('tenor','maturity')
        d=d[d[ten].astype(str).str.upper().str.replace(' ','').isin(['5Y','5YR','5YEAR','5YEARS','60M'])]
    d=d.rename(columns={dc:'date',cc:'country',sc:'cds5y_bp'})
    d['date']=pd.to_datetime(d['date'],errors='coerce',utc=True).dt.tz_localize(None)
    d['country']=d.country.map(standard_country)
    d['cds5y_bp']=pd.to_numeric(d.cds5y_bp,errors='coerce')
    d=d.dropna(subset=['date','cds5y_bp'])
    d=d[~d.country.str.startswith('UNKNOWN:')]
    d=d[d.cds5y_bp.between(0,MAX_CDS_BP,inclusive='left')]
    d=d.sort_values(['country','date']).drop_duplicates(['country','date'],keep='last')
    if d['date'].dt.to_period('M').nunique()<12:
        raise RuntimeError('Historical CDS covers fewer than 12 distinct months; likely a snapshot.')
    return d[['date','country','cds5y_bp']]

# Restore the frozen V2.3 duration proxy if it was not exported in the parent CSV.
# This is deterministic and does NOT rewrite the uploaded file or change the checked SHA.
def ensure_frozen_duration10(frame, source_label):
    frame=frame.copy()
    if 'duration10' not in frame.columns:
        y=pd.to_numeric(frame['y10'],errors='coerce').to_numpy(dtype=float)
        if not np.isfinite(y).all():
            raise ValueError(f'{source_label}: y10 must be finite to reconstruct duration10.')
        y=np.maximum(y,1e-4)
        frame['duration10']=(1.0-(1.0+y)**(-10))/y
        print(f'{source_label}: derived duration10 in memory with EXACT frozen V2.3 duration_proxy_10y(y10).')
    else:
        frame['duration10']=pd.to_numeric(frame['duration10'],errors='coerce')
    if not np.isfinite(frame['duration10'].to_numpy(dtype=float)).all() or (frame['duration10']<=0).any():
        raise ValueError(f'{source_label}: duration10 contains missing, non-finite or non-positive values.')
    return frame

PARENT_DURATION_DERIVED = 'duration10' not in pd.read_csv(MODEL_PATH,nrows=0).columns
model=pd.read_csv(MODEL_PATH,parse_dates=['date'])
model=ensure_frozen_duration10(model,'Frozen original panel')
model['country']=model.country.map(standard_country)
if model[['date','country']].duplicated().any():
    raise ValueError('Frozen panel has duplicate country-month keys')
if EXT_PATH is not None:
    extra=pd.read_csv(EXT_PATH,parse_dates=['date'])
    extra=ensure_frozen_duration10(extra,'Optional expanded panel')
    req=REQUIRED_PANEL|{'r3m'}
    if not req.issubset(extra.columns):
        raise ValueError(f'Expanded parent-model objects require: {sorted(req)}')
    extra['country']=extra.country.map(standard_country)
    if extra[['date','country']].duplicated().any():
        raise ValueError('Expanded panel contains duplicate country-month keys')
    overlapping=set(extra.country).intersection(model.country)
    if overlapping:
        raise ValueError('Expanded panel must contain NEW countries only, not rerun frozen countries: '+str(overlapping))
    model=pd.concat([model,extra],ignore_index=True)
    print('Externally reconstructed new MacroAnchor countries:',sorted(extra.country.unique()))
model['date']=model['date'].dt.to_period('M').dt.to_timestamp('M')
model=model[model.date<=PANEL_CUTOFF].sort_values(['country','date']).copy()
if model[['date','country']].duplicated().any():
    raise ValueError('Model panel duplicate keys after month-end normalization')
cds=read_cds(CDS_PATH)
print('Historical sovereign CDS:',len(cds),'daily observations across',cds.country.nunique(),'countries')

# Never use a quote later than the formation date; no extrapolation into unavailable spans.
sections=[]
for c,g in model.groupby('country',sort=True):
    h=cds.loc[cds.country==c,['date','cds5y_bp']].sort_values('date')
    g=g.sort_values('date').copy()
    if h.empty:
        g['cds5y_bp']=np.nan;g['cds_asof_date']=pd.NaT
    else:
        h=h.rename(columns={'date':'cds_asof_date'})
        g=pd.merge_asof(g,h,left_on='date',right_on='cds_asof_date',
            direction='backward', tolerance=pd.Timedelta(days=MAX_QUOTE_AGE_DAYS))
    sections.append(g)
panel=pd.concat(sections,ignore_index=True).sort_values(['country','date'])
valid=panel.cds5y_bp.notna()
assert (panel.loc[valid,'cds_asof_date']<=panel.loc[valid,'date']).all()
assert (panel.loc[valid,'date']-panel.loc[valid,'cds_asof_date'] <= pd.Timedelta(days=MAX_QUOTE_AGE_DAYS)).all()
coverage=(panel.groupby('country').agg(panel_months=('date','size'),months_with_cds=('cds5y_bp','count'),
                 first_month=('date','min'),last_month=('date','max')))
print('Coverage by country:')
display(coverage)
coverage.to_csv(OUT/'country_coverage.csv')


In [ ]:
# ===== Q PRICING: continuous-par CDS -> flat hazard; recovery-of-par defaultable ZC =====
# Flat deterministic discount r and Q hazard lambda, recovery R paid at default time:
#   P^D(0,T) = exp(-(r+lambda)*T) + R * lambda * Integral_0^T exp(-(r+lambda)*u) du
# This is a Q pricing identity under specified recovery convention, not P default loss.
# We invert P^D(0,10) = exp(-y_gov * 10) to obtain a credit-adjusted rates ZC proxy.
# Observed 10Y benchmark yields are coupon/par yields: the ZC step is a material approximation.

def cds5y_to_flat_hazard_q(cds_bp, recovery=RECOVERY_BASE):
    if not 0 <= recovery < 1: raise ValueError('Recovery must lie in [0,1).')
    if not np.isfinite(cds_bp) or cds_bp < 0: return np.nan
    s=float(cds_bp)/1e4
    return s/(1.0-recovery)

def defaultable_zc_price_q(r, hazard, recovery, maturity=TENOR_Y):
    if hazard < 0 or not (0 <= recovery < 1) or maturity <= 0:
        raise ValueError('Invalid hazard, recovery, or maturity')
    a=float(r)+float(hazard)
    surv=np.exp(-a*maturity)
    integral=maturity if abs(a)<1e-10 else -np.expm1(-a*maturity)/a
    return float(surv + recovery*hazard*integral)

def implied_credit_adjusted_yield_q(observed_yield, cds_bp, recovery=RECOVERY_BASE):
    if not np.isfinite(observed_yield) or not np.isfinite(cds_bp): return np.nan
    h=cds5y_to_flat_hazard_q(cds_bp,recovery)
    observed_price=float(np.exp(-float(observed_yield)*TENOR_Y))
    if observed_price <= 0: return np.nan
    def f(r):return defaultable_zc_price_q(r,h,recovery,TENOR_Y)-observed_price
    # Economically wide bracket supports negative sovereign yields (pre-2022 Europe/Japan).
    lo,hi=-1.0,1.0
    if f(lo)*f(hi)>0: return np.nan
    return float(brentq(f,lo,hi,xtol=1e-13))

# Unit/invariance tests before data-dependent inference.
for yg in (-0.01,0.0,0.02,0.08):
    assert abs(implied_credit_adjusted_yield_q(yg,0.0)-yg)<1e-11
for cds_test in (10.0,40.0,100.0,250.0):
    for r_test in (-0.01,0.02,0.06):
        haz=cds5y_to_flat_hazard_q(cds_test)
        observed=-np.log(defaultable_zc_price_q(r_test,haz,RECOVERY_BASE))/TENOR_Y
        recovered=implied_credit_adjusted_yield_q(observed,cds_test)
        assert abs(recovered-r_test)<1e-10
assert cds5y_to_flat_hazard_q(60.0,0.4)>cds5y_to_flat_hazard_q(30.0,0.4)
print('Q pricing round-trip and zero-CDS invariance tests: PASS')

# Pricing returns an inferred *rates Q-market yield proxy*; the subsequent TP compares it to
# the already-frozen P-expectations yield y_PE = y_gov - TP_raw.
def add_q_credit_signal(frame,recovery=RECOVERY_BASE,prefix='qcredit'):
    g=frame.copy()
    out_col=f'{prefix}_rates_yield_q'
    g[out_col]=[
        implied_credit_adjusted_yield_q(y,s,recovery)
        for y,s in zip(g.y10,g.cds5y_bp)
    ]
    g[f'{prefix}_hazard_q']=g.cds5y_bp.map(lambda s:cds5y_to_flat_hazard_q(s,recovery))
    g[f'{prefix}_survival_q_10y']=np.exp(-g[f'{prefix}_hazard_q']*TENOR_Y)
    g[f'{prefix}_pricing_component_bp']=(g.y10-g[out_col])*1e4
    g[f'{prefix}_tp_level']=g[out_col]-(g.y10-g.tp)
    # Share the frozen original country-level history for apples-to-apples scaling.
    # No future MacroAnchor data, including revisions to history inside notebook, are used.
    g[f'{prefix}_tp_z_past']=np.nan
    for c, grp in g.groupby('country',sort=False):
        idx=grp.sort_values('date').index
        a=g.loc[idx,'tp'].to_numpy(float)
        b=g.loc[idx,f'{prefix}_tp_level'].to_numpy(float)
        scores=np.full(len(idx),np.nan)
        for t in range(MIN_PARENT_HISTORY,len(idx)):
            hist=a[:t]
            hist=hist[np.isfinite(hist)]
            if len(hist)<MIN_PARENT_HISTORY or not np.isfinite(b[t]):continue
            sd=float(hist.std(ddof=1))
            if sd>1e-12:scores[t]=(b[t]-float(hist.mean()))/sd
        g.loc[idx,f'{prefix}_tp_z_past']=scores
    return g

panel=add_q_credit_signal(panel)
# Check that frozen raw standardization equals our past-only reconstruction (when available).
# Flag discrepancies rather than rewriting the parent signal.
raw_check=[]
for c,g in panel.groupby('country'):
    g=g.sort_values('date')
    a=g.tp.to_numpy(float); zz=[]
    for t in range(len(g)):
        hist=a[:t];hist=hist[np.isfinite(hist)]
        zz.append((a[t]-hist.mean())/hist.std(ddof=1) if t>=MIN_PARENT_HISTORY and
                  len(hist)>=MIN_PARENT_HISTORY and np.isfinite(a[t]) and hist.std(ddof=1)>1e-12 else np.nan)
    raw_check.extend(zip(g.index,np.asarray(zz,float)))
reconstructed=pd.Series(dict(raw_check)).reindex(panel.index).to_numpy(float)
masked=np.isfinite(reconstructed)&np.isfinite(panel.tp_z_past)
if masked.any():
    delta=float(np.max(np.abs(reconstructed[masked]-panel.tp_z_past.to_numpy(float)[masked])))
    print('Reconstructed raw historical z maximum discrepancy:',delta)
    if delta>1e-6:
        warnings.warn('Parent tp_z_past is NOT identical to reconstructed past-only z. Keep frozen parent score unchanged; investigate panel definition.')
print('Q-price-adjusted country-month observations:',int(panel.qcredit_tp_z_past.notna().sum()))


In [ ]:
# ===== FROZEN V2.3 TOP-3/BOTTOM-3, SAME-COUNTRY SAME-DATE DESIGN =====
def _rank_gross_weights(g,k=K,signal='tp_z_past'):
    if len(g)<max(MIN_COUNTRIES,2*k):return None
    g=g.dropna(subset=['duration10',signal]).sort_values([signal,'country']).copy()
    if len(g)<max(MIN_COUNTRIES,2*k):return None
    if (g.duration10<=0).any():return None
    q=np.zeros(len(g));q[:k]=-1/k;q[-k:]=1/k
    w=q/g.duration10.to_numpy(float)
    w=w/np.abs(w).sum()
    if abs(float(np.sum(w*g.duration10.to_numpy(float))))>1e-10:
        raise RuntimeError('DV01 neutrality invariant violated')
    g['weight']=w;g['dv01_weight']=w*g.duration10
    return g

def portfolio_panel(data,signal):
    rows=[];weights=[]
    for dt,g in data.groupby('date'):
        z=_rank_gross_weights(g,signal=signal)
        if z is None:continue
        if z.rx.isna().any():raise RuntimeError('Comparable universe has missing future return')
        rows.append({'date':dt,'ret12':float(np.sum(z.weight*z.rx)),
                     'net_dv01':float(z.dv01_weight.sum()),
                     'n_country_eligible':len(z),'n_positions':2*K})
        for _,r in z[z.weight!=0].iterrows():
            weights.append({'date':dt,'country':r.country,'weight':r.weight,
                            'dv01_weight':r.dv01_weight,'score':r[signal],
                            'cds5y_bp':r.cds5y_bp})
    result=pd.DataFrame(rows)
    w=pd.DataFrame(weights)
    if result.empty:return pd.DataFrame(columns=['ret12','net_dv01','n_country_eligible','n_positions']),w
    return result.set_index('date').sort_index(),w

def hac_mean(series,lag=NW_LAG):
    x=pd.Series(series).dropna().astype(float).to_numpy()
    if len(x)<30:return {'n':len(x),'mean_12m':float(x.mean()) if len(x) else np.nan,'t_NW12':np.nan}
    mu=x.mean();e=x-mu;S=float(e@e)
    for k in range(1,min(lag,len(x)-1)+1):
        S+=2*(1-k/(lag+1))*float(e[k:]@e[:-k])
    se=np.sqrt(max(S/(len(x)**2),0.0))
    return {'n':len(x),'mean_12m':float(mu),'t_NW12':float(mu/se) if se>0 else np.nan}

def weight_turnover(weights):
    if weights.empty:return pd.Series(dtype=float)
    w=weights.pivot_table(index='date',columns='country',values='weight',aggfunc='sum').fillna(0).sort_index()
    tr=.5*w.diff().abs().sum(axis=1)
    tr.iloc[0]=.5*w.iloc[0].abs().sum()
    return tr

# Exact same country observations for BOTH portfolios; no ex-post picking countries
# based on returns and no silent substitution of raw signal when Q credit is missing.
common=panel.dropna(subset=['tp_z_past','qcredit_tp_z_past','y10','tp','duration10','rx','cds5y_bp']).copy()
raw_pf,raw_w=portfolio_panel(common,'tp_z_past')
q_pf,q_w=portfolio_panel(common,'qcredit_tp_z_past')
assert raw_pf.index.equals(q_pf.index),'Raw and Q strategy dates differ!'
assert raw_pf.n_country_eligible.equals(q_pf.n_country_eligible)
if not raw_pf.empty:
    assert (raw_pf.net_dv01.abs()<1e-10).all()
    assert (q_pf.net_dv01.abs()<1e-10).all()

rows=[]
for name,p,w in [('Frozen MacroAnchor, eligible universe',raw_pf,raw_w),
                 ('Q-credit-adjusted MacroAnchor',q_pf,q_w)]:
    stats=hac_mean(p.ret12)
    to=weight_turnover(w)
    ann_to=float(12*to.reindex(p.index).mean()) if not p.empty else np.nan
    stats.update({'strategy':name,'annual_turnover_proxy':ann_to,
                  'net_mean_cost_5bp':stats['mean_12m']-ann_to*5/1e4 if not np.isnan(ann_to) else np.nan})
    rows.append(stats)
if not raw_pf.empty:
    paired=q_pf.ret12-raw_pf.ret12
    pair_stats=hac_mean(paired)
    rows.append({**pair_stats,'strategy':'Paired adjusted-minus-raw (same dates and countries)',
                 'annual_turnover_proxy':np.nan,'net_mean_cost_5bp':np.nan})
else:
    print('IMPORTANT: no month has >=6 countries with both frozen score and 5Y CDS. No L/S comparison possible; do not relax K or MIN_COUNTRIES silently.')
summary=pd.DataFrame(rows).set_index('strategy')
print('STRICT SAME-SAMPLE RESULTS (12m overlapping returns; NW(12)). This is a proxy backtest, not live P&L.')
display(summary)
print('Formed months:',len(raw_pf),'|',str(raw_pf.index.min()) if len(raw_pf) else 'NONE',
      'to',str(raw_pf.index.max()) if len(raw_pf) else 'NONE')

# Cross-sectional signal/ranking diagnostics. These also work when 6-country portfolios cannot form.
def rank_corr(g):
    if len(g)<3:return np.nan
    return g.tp_z_past.rank().corr(g.qcredit_tp_z_past.rank())
rank_by_month=common.groupby('date').apply(rank_corr,include_groups=False).rename('raw_vs_q_rank_spearman')
rank_summary=pd.DataFrame({'date':rank_by_month.index,'rank_spearman':rank_by_month.to_numpy()})
if len(raw_w) and len(q_w):
    wa=raw_w.pivot_table(index='date',columns='country',values='weight').fillna(0)
    wb=q_w.pivot_table(index='date',columns='country',values='weight').fillna(0)
    wa,wb=wa.align(wb,join='outer',axis=1,fill_value=0)
    aligned=(wa-wb).abs().sum(axis=1).rename('L1_weight_difference')
    print('Mean L1 change in gross-normalized weights:',float(aligned.mean()))
else:
    aligned=pd.Series(dtype=float,name='L1_weight_difference')
print('Median cross-country rank correlation:',rank_by_month.median())

# Same-sample cost grid. Costs are estimated on observed turnover; NOT execution prices.
cost_rows=[]
for name,p,w in [('Frozen MacroAnchor eligible',raw_pf,raw_w),
                 ('Q-credit-adjusted MacroAnchor',q_pf,q_w)]:
    turnover=weight_turnover(w)
    annual_to=float(12*turnover.reindex(p.index).mean()) if len(p) else np.nan
    for bp in COST_BP_GRID:
        cost_rows.append({'strategy':name,'cost_bp_per_one_way_unit':bp,
                         'annual_turnover_proxy':annual_to,
                         'gross_mean_12m':float(p.ret12.mean()) if len(p) else np.nan,
                         'net_mean_12m_estimate':float(p.ret12.mean()-annual_to*bp/1e4) if len(p) else np.nan})
cost_grid=pd.DataFrame(cost_rows)
print('Cost-grid sensitivity (same sample; approximate turnover model):')
display(cost_grid)

# Country-level pricing diagnostic. Not a hedged tradable return series.
price_diagnostics=panel[['date','country','cds_asof_date','cds5y_bp','y10','tp','tp_z_past',
    'qcredit_rates_yield_q','qcredit_pricing_component_bp','qcredit_tp_level',
    'qcredit_tp_z_past','qcredit_hazard_q','qcredit_survival_q_10y']].copy()


In [ ]:
# ===== ASSUMPTION DIAGNOSTICS: fixed recovery alternatives, no spec selection =====
sens_rows=[]
for R in RECOVERY_SENSITIVITIES:
    label=f'qcredit_R{int(round(R*100))}'
    alt=add_q_credit_signal(panel,recovery=R,prefix=label)
    score_col=label+'_tp_z_past'
    # Each scenario and its own frozen baseline must have identical dates and eligible countries.
    sub=alt.dropna(subset=['tp_z_past',score_col,'duration10','rx','cds5y_bp']).copy()
    raw_s,_=portfolio_panel(sub,'tp_z_past')
    adj_s,_=portfolio_panel(sub,score_col)
    if not raw_s.index.equals(adj_s.index):raise RuntimeError('Recovery scenario sample mismatch')
    sens_rows.append({'recovery_assumption':R,'n_common_months':len(raw_s),
                      'raw_mean':raw_s.ret12.mean() if len(raw_s) else np.nan,
                      'q_adjusted_mean':adj_s.ret12.mean() if len(adj_s) else np.nan,
                      'paired_difference_mean':(adj_s.ret12-raw_s.ret12).mean() if len(adj_s) else np.nan,
                      'paired_difference_t':hac_mean(adj_s.ret12-raw_s.ret12)['t_NW12'] if len(adj_s) else np.nan})
sensitivity=pd.DataFrame(sens_rows)
print('Sensitivity to recovery assumptions (NOT parameter search, NOT optimizations):')
display(sensitivity)

# Data provenance, explicit diagnostic nature of level correction, no Q->P substitution.
provenance={
    'version':'V4 Q sovereign-credit pricing audit — exploratory, no outcomes preclaimed',
    'panel_sha256':PANEL_SHA,'expected_frozen_panel_sha256':EXPECTED_PARENT_SHA,
    'parent_duration10_derived_in_memory':PARENT_DURATION_DERIVED,
    'duration10_method_if_missing':'frozen V2.3: D=(1-(1+max(y10,1e-4))**(-10))/max(y10,1e-4)',
    'cds_input_sha256':file_hash(CDS_PATH),'cds_input_file':str(CDS_PATH),
    'input_filenames':UPLOADED_NAMES,
    'Q_hazard_calibration':'flat continuous par CDS s=(1-R)*lambda_Q, 5Y level extrapolated to 10Y',
    'bond_Q_pricing':'zero-coupon, recovery of face at default, constant hazard and flat riskfree yield',
    'recovery_primary':RECOVERY_BASE,'recovery_diagnostics':RECOVERY_SENSITIVITIES,
    'future_return_target':'frozen V2.3 rx duration-approximated 12m realized local-currency proxy',
    'common_country_dates':True,'signal_base':'past original MacroAnchor TP country moments, 120-month minimum',
    'cds_staleness_days':MAX_QUOTE_AGE_DAYS,
    'unsupported_claims':['rates-only realized returns','hedged CS01','hedged FX P&L',
       'P default probabilities','10Y CDS term structure identified from 5Y','observed cash-bond CDS basis neutralized'],
}
(OUT/'v4_q_pricing_provenance.json').write_text(json.dumps(provenance,indent=2,default=str))
summary.to_csv(OUT/'v4_common_sample_summary.csv')
cost_grid.to_csv(OUT/'v4_same_sample_cost_grid.csv',index=False)
price_diagnostics.to_csv(OUT/'v4_country_month_q_pricing_diagnostics.csv',index=False)
coverage.to_csv(OUT/'v4_country_coverage.csv')
rank_summary.to_csv(OUT/'v4_rank_diagnostics.csv',index=False)
sensitivity.to_csv(OUT/'v4_recovery_sensitivity.csv',index=False)
if not raw_pf.empty:
    joined=pd.DataFrame({'raw_ret12':raw_pf.ret12,'qcredit_ret12':q_pf.ret12,
                         'q_minus_raw':q_pf.ret12-raw_pf.ret12})
    joined.to_csv(OUT/'v4_paired_returns.csv')
    raw_w.to_csv(OUT/'v4_raw_weights.csv',index=False)
    q_w.to_csv(OUT/'v4_qcredit_weights.csv',index=False)
# Export only source-free derived diagnostics; do not redistribute proprietary raw CDS data.
print('Research outputs written to:',OUT.resolve())
# One final results archive, prepared unattended; does not trigger any browser dialog.
import shutil
from IPython.display import FileLink
results_zip = shutil.make_archive('macroanchor_qcredit_v4_results','zip',OUT)
print('Download-ready V4 diagnostics archive:')
display(FileLink(results_zip))


## Interpretation gate — do not promote ex-post best recovery assumption

1. **No eligible portfolios:** the files do not contain the breadth required by the frozen top-3/bottom-3 strategy. Do **not** drop the constraint opportunistically.
2. **Q-adjusted signal fails:** informative, but cannot isolate whether this is due to a noisy CDS proxy, reference-currency mismatch, sovereign CDS/bond basis, a missing 10Y CDS curve, or genuine signal deterioration.
3. **Q-adjusted signal succeeds:** promising but **not proof of rates-only alpha**. Both strategies still earn the *original defaultable government bond return proxy*, with no actual CDS hedge, no FX forward trading, and no institution-grade accrued/coupon bond pricing.
4. **Paired delta:** interpreted only on the exact common country/date set; a favorable result is exploratory because this version follows earlier research.

### Next strict model upgrade (not implemented in V4)

Actual 5Y/10Y sovereign CDS **term structure** + contract/currency alignment; coupon-bearing bonds and risk-free OIS discount curves; joint rates/credit default dependence; market quotes under Q and independently estimated P intensity (historical migration/default), with a proper P-vs-Q risk premium decomposition and CS01/FX hedging. Do not infer P intensities from CDS alone.
